# Complete Mininet PCAP to ML Model Pipeline

This notebook provides end-to-end training from PCAP files to production-ready ML models.

**Pipeline:**
1. Load and parse PCAP files
2. Extract network flow features
3. Preprocess and engineer features
4. Train multiple ML models
5. Comprehensive evaluation with all metrics
6. Save production-ready models

**Requirements:**
- PCAP files from Mininet
- Python 3.8+
- Libraries: scapy, pandas, scikit-learn, xgboost, imbalanced-learn

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -q scapy pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn

## Step 2: Import Libraries

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict
import joblib

# Scapy for PCAP parsing
from scapy.all import rdpcap, IP, TCP, UDP, ICMP

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

# XGBoost
import xgboost as xgb

# SMOTE for class imbalance
from imblearn.over_sampling import SMOTE

print("✓ All libraries imported successfully")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Step 3: Feature Extraction from PCAP Files

In [ ]:
class PCAPFeatureExtractor:
    """Extract ML features from PCAP files"""
    
    def __init__(self):
        self.features = []
    
    def extract_from_pcap(self, pcap_file, label, attack_type):
        """Extract features from a single PCAP file"""
        print(f"\nProcessing: {os.path.basename(pcap_file)}")
        print(f"  Label: {label} ({attack_type})")
        
        try:
            packets = rdpcap(pcap_file)
            print(f"  Total packets: {len(packets)}")
        except Exception as e:
            print(f"  ❌ Error reading PCAP: {e}")
            return []
        
        # Group packets by flow
        flows = defaultdict(list)
        
        for pkt in packets:
            if IP in pkt:
                flow_key = self._get_flow_key(pkt)
                if flow_key:
                    flows[flow_key].append(pkt)
        
        print(f"  Flows identified: {len(flows)}")
        
        # Extract features for each flow
        flow_features = []
        for flow_key, flow_packets in flows.items():
            feature = self._extract_flow_features(flow_key, flow_packets, label, attack_type)
            if feature:
                flow_features.append(feature)
        
        print(f"  ✓ Extracted {len(flow_features)} flow features")
        return flow_features
    
    def _get_flow_key(self, pkt):
        """Get flow identifier from packet"""
        if IP not in pkt:
            return None
        
        src_ip = pkt[IP].src
        dst_ip = pkt[IP].dst
        
        if TCP in pkt:
            protocol = 'TCP'
            src_port = pkt[TCP].sport
            dst_port = pkt[TCP].dport
        elif UDP in pkt:
            protocol = 'UDP'
            src_port = pkt[UDP].sport
            dst_port = pkt[UDP].dport
        elif ICMP in pkt:
            protocol = 'ICMP'
            src_port = 0
            dst_port = 0
        else:
            return None
        
        return (src_ip, dst_ip, src_port, dst_port, protocol)
    
    def _extract_flow_features(self, flow_key, packets, label, attack_type):
        """Extract comprehensive features from flow packets"""
        src_ip, dst_ip, src_port, dst_port, protocol = flow_key
        
        packet_count = len(packets)
        if packet_count == 0:
            return None
        
        # Timing features
        timestamps = [float(pkt.time) for pkt in packets]
        duration = max(timestamps) - min(timestamps) if len(timestamps) > 1 else 0.001
        
        # Size features
        packet_sizes = [len(pkt) for pkt in packets]
        byte_count = sum(packet_sizes)
        
        # TCP flags
        syn_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x02)
        fin_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x01)
        rst_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x04)
        psh_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x08)
        ack_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x10)
        urg_count = sum(1 for pkt in packets if TCP in pkt and pkt[TCP].flags & 0x20)
        
        # Derived features
        packets_per_sec = packet_count / duration
        bytes_per_sec = byte_count / duration
        mean_packet_size = np.mean(packet_sizes)
        std_packet_size = np.std(packet_sizes) if len(packet_sizes) > 1 else 0
        min_packet_size = min(packet_sizes)
        max_packet_size = max(packet_sizes)
        
        # Inter-arrival times
        if len(timestamps) > 1:
            inter_arrival_times = np.diff(timestamps)
            mean_iat = np.mean(inter_arrival_times)
            std_iat = np.std(inter_arrival_times)
        else:
            mean_iat = 0
            std_iat = 0
        
        # Flag ratios
        syn_ratio = syn_count / packet_count
        fin_ratio = fin_count / packet_count
        rst_ratio = rst_count / packet_count
        psh_ratio = psh_count / packet_count
        ack_ratio = ack_count / packet_count
        
        # Port features
        is_well_known_port = 1 if dst_port < 1024 else 0
        
        return {
            'duration': duration,
            'protocol': protocol,
            'src_port': src_port,
            'dst_port': dst_port,
            'packet_count': packet_count,
            'byte_count': byte_count,
            'packets_per_sec': packets_per_sec,
            'bytes_per_sec': bytes_per_sec,
            'mean_packet_size': mean_packet_size,
            'std_packet_size': std_packet_size,
            'min_packet_size': min_packet_size,
            'max_packet_size': max_packet_size,
            'mean_inter_arrival_time': mean_iat,
            'std_inter_arrival_time': std_iat,
            'syn_count': syn_count,
            'fin_count': fin_count,
            'rst_count': rst_count,
            'psh_count': psh_count,
            'ack_count': ack_count,
            'urg_count': urg_count,
            'syn_ratio': syn_ratio,
            'fin_ratio': fin_ratio,
            'rst_ratio': rst_ratio,
            'psh_ratio': psh_ratio,
            'ack_ratio': ack_ratio,
            'is_well_known_port': is_well_known_port,
            'label': label,
            'attack_type': attack_type
        }

print("✓ Feature extractor class defined")

## Step 4: Load and Process PCAP Files

In [ ]:
print("="*60)
print("LOADING AND PROCESSING PCAP FILES")
print("="*60)

# Initialize extractor
extractor = PCAPFeatureExtractor()

# Define PCAP files and their labels
pcap_files = [
    # Normal traffic
    ('../data_capture/pcaps/normal_traffic_20251008_003311.pcap', 0, 'normal'),
    
    # Attack traffic
    ('data_capture/mininet/syn_flood.pcap', 1, 'syn_flood'),
    ('data_capture/mininet/port_scan.pcap', 1, 'port_scan'),
    ('data_capture/mininet/udp_flood.pcap', 1, 'udp_flood'),
    ('data_capture/mininet/http_flood.pcap', 1, 'http_flood'),
]

# Extract features from all PCAP files
all_features = []

for pcap_file, label, attack_type in pcap_files:
    if os.path.exists(pcap_file):
        features = extractor.extract_from_pcap(pcap_file, label, attack_type)
        all_features.extend(features)
    else:
        print(f"\n⚠ Warning: {pcap_file} not found, skipping...")

# Convert to DataFrame
df = pd.DataFrame(all_features)

print(f"\n{'='*60}")
print(f"✓ Total samples extracted: {len(df):,}")
print(f"  Normal: {len(df[df['label'] == 0]):,}")
print(f"  Attack: {len(df[df['label'] == 1]):,}")
print(f"  Features: {len(df.columns)}")
print("="*60)

# Display first few rows
df.head()

## Step 5: Data Analysis and Visualization

In [ ]:
print("\n" + "="*60)
print("DATA ANALYSIS")
print("="*60)

# Class distribution
print("\nClass Distribution:")
print(df['label'].value_counts())
print(f"\nNormal: {len(df[df['label'] == 0])} ({len(df[df['label'] == 0])/len(df)*100:.1f}%)")
print(f"Attack: {len(df[df['label'] == 1])} ({len(df[df['label'] == 1])/len(df)*100:.1f}%)")

# Attack type distribution
print("\nAttack Type Distribution:")
attack_counts = df['attack_type'].value_counts()
for attack_type, count in attack_counts.items():
    print(f"  {attack_type}: {count}")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Class distribution
ax = axes[0, 0]
df['label'].value_counts().plot(kind='bar', ax=ax, color=['green', 'red'])
ax.set_title('Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
ax.set_xticklabels(['Normal', 'Attack'], rotation=0)

# Attack type distribution
ax = axes[0, 1]
attack_counts.plot(kind='bar', ax=ax, color='coral')
ax.set_title('Attack Type Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Attack Type')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# Packets per second distribution
ax = axes[1, 0]
df[df['label'] == 0]['packets_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Normal', color='green')
df[df['label'] == 1]['packets_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Attack', color='red')
ax.set_title('Packets Per Second Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Packets/sec')
ax.set_ylabel('Frequency')
ax.legend()
ax.set_xlim(0, df['packets_per_sec'].quantile(0.95))

# Bytes per second distribution
ax = axes[1, 1]
df[df['label'] == 0]['bytes_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Normal', color='green')
df[df['label'] == 1]['bytes_per_sec'].hist(bins=50, ax=ax, alpha=0.7, label='Attack', color='red')
ax.set_title('Bytes Per Second Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Bytes/sec')
ax.set_ylabel('Frequency')
ax.legend()
ax.set_xlim(0, df['bytes_per_sec'].quantile(0.95))

plt.tight_layout()
plt.savefig('data_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualizations saved to data_analysis.png")

## Step 6: Data Preprocessing

In [ ]:
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Separate features and labels
X = df.drop(['label', 'attack_type'], axis=1)
y = df['label']
attack_types = df['attack_type']

# Encode categorical features
print("\nEncoding categorical features...")
label_encoders = {}
for col in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le
    print(f"  Encoded: {col}")

# Handle missing and infinite values
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

print(f"\n✓ Features: {len(X.columns)}")
print(f"✓ Samples: {len(X)}")
print(f"✓ Normal: {sum(y == 0)}, Attack: {sum(y == 1)}")

# Feature list
print("\nFeature List:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i}. {col}")

## Step 7: Train/Validation/Test Split

In [ ]:
print("\n" + "="*60)
print("DATA SPLITTING")
print("="*60)

# Split: 60% train, 20% validation, 20% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nTrain Set: {len(X_train)} samples")
print(f"  Normal: {sum(y_train == 0)}, Attack: {sum(y_train == 1)}")

print(f"\nValidation Set: {len(X_val)} samples")
print(f"  Normal: {sum(y_val == 0)}, Attack: {sum(y_val == 1)}")

print(f"\nTest Set: {len(X_test)} samples")
print(f"  Normal: {sum(y_test == 0)}, Attack: {sum(y_test == 1)}")

# Visualize split
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (y_split, title) in zip(axes, [(y_train, 'Train'), (y_val, 'Validation'), (y_test, 'Test')]):
    counts = y_split.value_counts()
    ax.bar(['Normal', 'Attack'], [counts.get(0, 0), counts.get(1, 0)], color=['green', 'red'])
    ax.set_title(f'{title} Set', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig('data_split.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Data split visualization saved")

## Step 8: Feature Scaling and Selection

In [ ]:
print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
print("✓ Features scaled")

# Feature selection
print("\nSelecting top features...")
k_features = min(30, X_train.shape[1])
selector = SelectKBest(mutual_info_classif, k=k_features)
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"✓ Selected {len(selected_features)} features")

# Feature importance scores
feature_scores = pd.DataFrame({
    'feature': X.columns,
    'score': selector.scores_
}).sort_values('score', ascending=False)

print("\nTop 10 Features by Importance:")
for i, row in feature_scores.head(10).iterrows():
    print(f"  {row['feature']}: {row['score']:.4f}")

# Visualize feature importance
plt.figure(figsize=(12, 6))
top_features = feature_scores.head(20)
plt.barh(range(len(top_features)), top_features['score'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance Score', fontsize=12)
plt.title('Top 20 Feature Importance Scores', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance visualization saved")

## Step 9: Handle Class Imbalance with SMOTE

In [ ]:
print("\n" + "="*60)
print("CLASS BALANCING WITH SMOTE")
print("="*60)

print(f"\nBefore SMOTE:")
print(f"  Total: {len(X_train_selected)}")
print(f"  Normal: {sum(y_train == 0)}")
print(f"  Attack: {sum(y_train == 1)}")
print(f"  Ratio: {sum(y_train == 0) / sum(y_train == 1):.2f}:1")

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)

print(f"\nAfter SMOTE:")
print(f"  Total: {len(X_train_balanced)}")
print(f"  Normal: {sum(y_train_balanced == 0)}")
print(f"  Attack: {sum(y_train_balanced == 1)}")
print(f"  Ratio: 1:1 (balanced)")

# Visualize balancing
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(['Normal', 'Attack'], [sum(y_train == 0), sum(y_train == 1)], color=['green', 'red'])
ax1.set_title('Before SMOTE', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count')

ax2.bar(['Normal', 'Attack'], [sum(y_train_balanced == 0), sum(y_train_balanced == 1)], color=['green', 'red'])
ax2.set_title('After SMOTE', fontsize=12, fontweight='bold')
ax2.set_ylabel('Count')

plt.tight_layout()
plt.savefig('smote_balancing.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ SMOTE balancing visualization saved")

## Step 10: Train Random Forest Model

In [ ]:
import time

print("\n" + "="*60)
print("TRAINING RANDOM FOREST")
print("="*60)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("\nTraining Random Forest...")
start_time = time.time()
rf_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.2f} seconds")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_train_balanced, y_train_balanced, cv=cv, scoring='f1', n_jobs=-1)
print(f"CV F1 Scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Validation set performance
val_score = rf_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 11: Train XGBoost Model

In [ ]:
print("\n" + "="*60)
print("TRAINING XGBOOST")
print("="*60)

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

print("\nTraining XGBoost...")
start_time = time.time()
xgb_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Training completed in {training_time:.2f} seconds")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=cv, scoring='f1', n_jobs=-1)
print(f"CV F1 Scores: {cv_scores}")
print(f"Mean CV F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Validation set performance
val_score = xgb_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 12: Create Ensemble Model

In [ ]:
print("\n" + "="*60)
print("CREATING ENSEMBLE")
print("="*60)

ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('xgb', xgb_model)
    ],
    voting='soft',
    n_jobs=-1
)

print("\nTraining ensemble...")
start_time = time.time()
ensemble_model.fit(X_train_balanced, y_train_balanced)
training_time = time.time() - start_time
print(f"✓ Ensemble created in {training_time:.2f} seconds")

# Validation set performance
val_score = ensemble_model.score(X_val_selected, y_val)
print(f"\nValidation Accuracy: {val_score:.4f}")

## Step 13: Comprehensive Model Evaluation

In [ ]:
print("\n" + "="*60)
print("MODEL EVALUATION ON TEST SET")
print("="*60)

# Predictions
y_pred_rf = rf_model.predict(X_test_selected)
y_pred_xgb = xgb_model.predict(X_test_selected)
y_pred_ensemble = ensemble_model.predict(X_test_selected)

y_pred_proba_rf = rf_model.predict_proba(X_test_selected)[:, 1]
y_pred_proba_xgb = xgb_model.predict_proba(X_test_selected)[:, 1]
y_pred_proba_ensemble = ensemble_model.predict_proba(X_test_selected)[:, 1]

# Calculate metrics for all models
models = {
    'Random Forest': (y_pred_rf, y_pred_proba_rf),
    'XGBoost': (y_pred_xgb, y_pred_proba_xgb),
    'Ensemble': (y_pred_ensemble, y_pred_proba_ensemble)
}

results = {}
for name, (y_pred, y_pred_proba) in models.items():
    results[name] = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_pred_proba)
    }

# Display results
results_df = pd.DataFrame(results).T
print("\nModel Performance Comparison:")
print(results_df.to_string())

# Detailed report for ensemble
print("\n" + "="*60)
print("ENSEMBLE MODEL - DETAILED METRICS")
print("="*60)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble, target_names=['Normal', 'Attack'], digits=4))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_ensemble)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

# Calculate additional metrics
specificity = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
sensitivity = cm[1,1] / (cm[1,1] + cm[1,0]) if (cm[1,1] + cm[1,0]) > 0 else 0

print(f"\nSpecificity (True Negative Rate): {specificity:.4f}")
print(f"Sensitivity (True Positive Rate/Recall): {sensitivity:.4f}")

## Step 14: Comprehensive Visualizations

In [ ]:
# 1. Confusion Matrices for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, (y_pred, _)) in zip(axes, models.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'])
    ax.set_title(f'{name}\nAccuracy: {accuracy_score(y_test, y_pred):.4f}',
                fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. ROC Curves
plt.figure(figsize=(10, 8))

for name, (_, y_pred_proba) in models.items():
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=2)
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 3. Precision-Recall Curves
plt.figure(figsize=(10, 8))

for name, (_, y_pred_proba) in models.items():
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    plt.plot(recall, precision, label=f'{name} (AP = {ap:.4f})', linewidth=2)

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves - Model Comparison', fontsize=14, fontweight='bold')
plt.legend(loc='lower left', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('precision_recall_curves.png', dpi=300, bbox_inches='tight')
plt.show()

# 4. Model Performance Comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC']

for ax, metric, metric_name in zip(axes.flat, metrics, metric_names):
    values = [results[model][metric] for model in results.keys()]
    bars = ax.bar(results.keys(), values, color=['skyblue', 'lightcoral', 'lightgreen'])
    ax.set_title(metric_name, fontsize=12, fontweight='bold')
    ax.set_ylim([0, 1.1])
    ax.set_ylabel('Score')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10)

# Hide the last subplot
axes.flat[-1].axis('off')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ All visualizations generated and saved!")

## Step 15: Save Models and Artifacts

In [ ]:
print("\n" + "="*60)
print("SAVING MODELS AND ARTIFACTS")
print("="*60)

# Create output directory
output_dir = '../models'
os.makedirs(output_dir, exist_ok=True)

# Save models
print("\nSaving models...")
joblib.dump(ensemble_model, f'{output_dir}/mininet_ensemble_model.pkl')
joblib.dump(rf_model, f'{output_dir}/mininet_random_forest_model.pkl')
joblib.dump(xgb_model, f'{output_dir}/mininet_xgboost_model.pkl')
print("  ✓ Models saved")

# Save preprocessing components
print("\nSaving preprocessing components...")
joblib.dump(scaler, f'{output_dir}/mininet_scaler.pkl')
joblib.dump(selector, f'{output_dir}/mininet_feature_selector.pkl')
joblib.dump(selected_features, f'{output_dir}/mininet_feature_columns.pkl')
joblib.dump(label_encoders, f'{output_dir}/mininet_label_encoders.pkl')
print("  ✓ Preprocessing components saved")

# Save metadata
print("\nSaving metadata...")
metadata = {
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'n_normal': int(sum(df['label'] == 0)),
    'n_attack': int(sum(df['label'] == 1)),
    'n_features': len(selected_features),
    'selected_features': selected_features,
    'ensemble_metrics': {
        'accuracy': float(results['Ensemble']['accuracy']),
        'precision': float(results['Ensemble']['precision']),
        'recall': float(results['Ensemble']['recall']),
        'f1_score': float(results['Ensemble']['f1']),
        'roc_auc': float(results['Ensemble']['roc_auc'])
    },
    'confusion_matrix': {
        'true_negatives': int(cm[0,0]),
        'false_positives': int(cm[0,1]),
        'false_negatives': int(cm[1,0]),
        'true_positives': int(cm[1,1])
    }
}
joblib.dump(metadata, f'{output_dir}/mininet_model_metadata.pkl')
print("  ✓ Metadata saved")

print("\n" + "="*60)
print("✓ ALL MODELS AND ARTIFACTS SAVED")
print("="*60)
print(f"\nOutput directory: {output_dir}")
print("\nFiles saved:")
print("  1. mininet_ensemble_model.pkl")
print("  2. mininet_random_forest_model.pkl")
print("  3. mininet_xgboost_model.pkl")
print("  4. mininet_scaler.pkl")
print("  5. mininet_feature_selector.pkl")
print("  6. mininet_feature_columns.pkl")
print("  7. mininet_label_encoders.pkl")
print("  8. mininet_model_metadata.pkl")
print("\n" + "="*60)

## Step 16: Final Summary Report

In [ ]:
print("\n" + "="*60)
print("TRAINING COMPLETE - FINAL SUMMARY")
print("="*60)

print(f"\n📊 Dataset:")
print(f"  Total Samples: {len(df):,}")
print(f"  Normal: {sum(df['label'] == 0):,}")
print(f"  Attack: {sum(df['label'] == 1):,}")
print(f"  Features: {len(selected_features)}")

print(f"\n🎯 Best Model (Ensemble):")
print(f"  Accuracy:  {results['Ensemble']['accuracy']:.4f}")
print(f"  Precision: {results['Ensemble']['precision']:.4f}")
print(f"  Recall:    {results['Ensemble']['recall']:.4f}")
print(f"  F1-Score:  {results['Ensemble']['f1']:.4f}")
print(f"  ROC AUC:   {results['Ensemble']['roc_auc']:.4f}")

print(f"\n📁 Output Files:")
print(f"  Models: {output_dir}/")
print(f"  Visualizations: Current directory")

print(f"\n🎉 SUCCESS!")
print(f"  Models trained from real Mininet PCAP files")
print(f"  Ready for deployment to SOC dashboard")
print("\n" + "="*60)

# Create summary DataFrame
summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC AUC'],
    'Random Forest': [results['Random Forest'][m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']],
    'XGBoost': [results['XGBoost'][m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']],
    'Ensemble': [results['Ensemble'][m] for m in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
})

print("\n📊 Complete Performance Summary:")
print(summary_df.to_string(index=False))

print("\n✅ Notebook execution complete!")